# MES – Hypothesis 4: Volume Spike Reversal (long after high volume + negative return)

**Idea**: When volume z-score > 1.5 and previous minute return < 0, buy expecting reversal.  

**Exit**: After N minutes.  

**Constraint**: Regular hours only (8:30‑16:00 ET).

## 1. Setup and load data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import os
import warnings
warnings.filterwarnings('ignore')

IS_PATH = "../../../../Local-Data/processed/MES_IS_1min.csv"
OOS_TICKS_PATH = "../../../../Local-Data/processed/MES_OOS_ticks.csv"

df_is = pd.read_csv(IS_PATH, parse_dates=['DateTime'], index_col='DateTime')
print(f"IS data: {len(df_is):,} rows")
df_is.head()

IS data: 87,799 rows


,Open,High,Low,Close,Volume
DateTime,,,,,
2026-02-01 13:52:00,6961.25,6961.25,6961.25,6961.25,1
2026-02-01 20:01:00,6939.50,6945.25,6928.25,6936.75,2717
2026-02-01 20:02:00,6937.25,6937.75,6925.75,6926.75,1939
2026-02-01 20:03:00,6926.50,6927.75,6920.00,6927.00,2748
2026-02-01 20:04:00,6927.00,6931.50,6926.00,6928.75,1374


## 2. Feature engineering (volume z-score, ATR, returns)

In [4]:
def add_features(df):
    df = df.copy()
    df['return'] = df['Close'].pct_change()
    df['volume_sma'] = df['Volume'].rolling(20).mean()
    df['volume_std'] = df['Volume'].rolling(20).std()
    df['volume_zscore'] = (df['Volume'] - df['volume_sma']) / df['volume_std']
    tr = pd.concat([df['High'] - df['Low'],
                    (df['High'] - df['Close'].shift()).abs(),
                    (df['Low'] - df['Close'].shift()).abs()], axis=1).max(axis=1)
    df['ATR14'] = tr.rolling(14).mean()
    return df

df_is = add_features(df_is).dropna()
print(f"IS after features: {len(df_is):,} rows")


IS after features: 87,780 rows


## 3. Restrict to regular trading hours

In [5]:
start_time = pd.Timestamp('2026-02-01 08:30:00').time()
end_time   = pd.Timestamp('2026-02-01 16:00:00').time()
df_is_reg = df_is.between_time(start_time, end_time).copy()
print(f"IS regular hours rows: {len(df_is_reg):,}")

IS regular hours rows: 28,459


## 4. Backtest function for volume spike reversal

In [6]:
def backtest_volume_reversal(df, vol_zscore_thresh=1.5, hold_minutes=10, require_neg_return=True):
    """
    Long when volume_zscore > threshold and (optionally) previous return < 0.
    Exit after hold_minutes minutes.
    """
    cond_vol = df['volume_zscore'] > vol_zscore_thresh
    cond_ret = df['return'].shift(1) < 0 if require_neg_return else True
    signal = cond_vol & cond_ret
    trades = []
    for idx in df.index[signal]:
        entry_price = df.loc[idx, 'Open']
        pos = df.index.get_loc(idx)
        if pos + hold_minutes >= len(df):
            continue
        exit_idx = df.index[pos + hold_minutes]
        exit_price = df.loc[exit_idx, 'Close']
        pnl_points = exit_price - entry_price
        trades.append(pnl_points * 5.0)   # USD
    return trades

## 5. Grid search on IS (vol threshold and hold minutes)

In [7]:
vol_thresholds = [1.2, 1.5, 1.8, 2.0]
hold_minutes = [5, 10, 15, 20]
results = []
for vt in vol_thresholds:
    for hm in hold_minutes:
        trades = backtest_volume_reversal(df_is_reg, vt, hm, require_neg_return=True)
        if trades:
            total = sum(trades)
            wins = sum(1 for t in trades if t > 0)
            win_rate = wins / len(trades)
            gross_profit = sum(t for t in trades if t > 0)
            gross_loss = -sum(t for t in trades if t < 0)
            pf = gross_profit / gross_loss if gross_loss != 0 else 0
            results.append({'vol_th': vt, 'hold': hm, 'trades': len(trades),
                            'pnl_usd': total, 'win_rate': win_rate, 'profit_factor': pf})
results_df = pd.DataFrame(results).sort_values('profit_factor', ascending=False)
print("Top 5 parameter sets on IS (regular hours):")
print(results_df.head(5).to_string(index=False))

Top 5 parameter sets on IS (regular hours):
 vol_th  hold  trades  pnl_usd  win_rate  profit_factor
    1.8    20     977   756.25  0.457523       1.029506
    1.2    20    1787   697.50  0.475658       1.015332
    1.5    20    1310   317.50  0.463359       1.009296
    1.2    15    1787 -2480.00  0.466144       0.939257
    1.8    15     977 -2193.75  0.447288       0.905089


## 6. Best parameters

In [8]:
best = results_df.iloc[0] if not results_df.empty else None
if best is not None:
    print(f"\nBest: vol_threshold={best['vol_th']}, hold={best['hold']}, trades={best['trades']}, PF={best['profit_factor']:.2f}")


Best: vol_threshold=1.8, hold=20.0, trades=977.0, PF=1.03


## 7. Validate on OOS (May 2026) using aggregated ticks

In [9]:
if os.path.exists(OOS_TICKS_PATH) and best is not None:
    df_ticks = pd.read_csv(OOS_TICKS_PATH, parse_dates=['DateTime'], index_col='DateTime')
    ohlc = df_ticks['Price'].resample('1min').ohlc()
    vol = df_ticks['Volume'].resample('1min').sum()
    ohlc.columns = ['Open', 'High', 'Low', 'Close']
    df_oos = ohlc.join(vol).rename(columns={'Volume': 'Volume'}).dropna()
    df_oos = add_features(df_oos).dropna()
    df_oos_reg = df_oos.between_time(start_time, end_time).copy()
    print(f"OOS regular hours rows: {len(df_oos_reg):,}")
    
    trades_oos = backtest_volume_reversal(df_oos_reg, best['vol_th'], int(best['hold']), require_neg_return=True)
    if trades_oos:
        total = sum(trades_oos)
        wins = sum(1 for t in trades_oos if t > 0)
        win_rate = wins / len(trades_oos)
        gross_profit = sum(t for t in trades_oos if t > 0)
        gross_loss = -sum(t for t in trades_oos if t < 0)
        pf = gross_profit / gross_loss if gross_loss != 0 else 0
        print("\n=== OOS VALIDATION ===")
        print(f"Trades: {len(trades_oos)}")
        print(f"Win rate: {win_rate:.2%}")
        print(f"Profit factor: {pf:.2f}")
        print(f"Net profit (USD): ${total:.2f}")
        if pf > 1.2:
            print("✅ Hypothesis 4 passes validation.")
        else:
            print("❌ Hypothesis 4 fails (PF <= 1.2).")
    else:
        print("No trades in OOS.")
else:
    print("OOS ticks file not found or no best parameters.")

OOS regular hours rows: 7,667

=== OOS VALIDATION ===
Trades: 217
Win rate: 55.76%
Profit factor: 1.08
Net profit (USD): $353.75
❌ Hypothesis 4 fails (PF <= 1.2).
